# 计算态演化

In [2]:
import quante as qt
L = 12
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)
state = qt.generate.state.random(basis.Ns, seed=42)
qt.linalg.expm_multiply(mat, state, -1j, start=0, stop=10, num=100, herm=True)

array([[[ 0.00335624+0.00606165j],
        [-0.01145467+0.0090065j ],
        [ 0.00826567+0.0048053j ],
        ...,
        [-0.01305153+0.00220341j],
        [ 0.00607938+0.00661749j],
        [-0.01943596+0.01879979j]],

       [[ 0.0048898 +0.00490894j],
        [-0.00893959+0.01087129j],
        [ 0.01002091+0.00476051j],
        ...,
        [-0.01273198+0.00414389j],
        [ 0.00764766+0.00570824j],
        [-0.01353566+0.02340886j]],

       [[ 0.00604849+0.00337989j],
        [-0.00609528+0.01202976j],
        [ 0.01195179+0.00416268j],
        ...,
        [-0.01204725+0.00597486j],
        [ 0.00906296+0.00443277j],
        [-0.00659764+0.02622327j]],

       ...,

       [[ 0.00508586-0.00470552j],
        [ 0.0022496 +0.01370704j],
        [-0.00079423+0.0112147j ],
        ...,
        [ 0.00022375+0.00682789j],
        [ 0.00216894-0.0012144j ],
        [ 0.02289399+0.01438937j]],

       [[ 0.00360056-0.00591979j],
        [ 0.0058218 +0.01275967j],
        [ 0.00174

In [1]:
import quante as qt
import numpy as np
import scipy.sparse as sp
import time

## 初始化态和哈密顿量

In [2]:
# 拿到矩阵
L = 12
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

## 比较不同方法的计算结果

### 用本征分解的方法计算

In [4]:
res = qt.linalg.eigh(mat)
times = np.linspace(0,10,100)
qt.linalg.get_time_evolution_states_ED(state,*res,times, device_name='cpu')

array([[ 0.00335624+0.00606165j,  0.0048898 +0.00490894j,  0.00604849+0.00337989j, ...,  0.00508586-0.00470552j,  0.00360056-0.00591979j,  0.00183922-0.00668021j],
       [-0.01145467+0.0090065j , -0.00893959+0.01087129j, -0.00609528+0.01202976j, ...,  0.0022496 +0.01370704j,  0.0058218 +0.01275967j,  0.00901401+0.01092018j],
       [ 0.00826567+0.0048053j ,  0.01002091+0.00476051j,  0.01195179+0.00416268j, ..., -0.00079423+0.0112147j ,  0.00174312+0.01032498j,  0.00388811+0.00880588j],
       ...,
       [-0.01305153+0.00220341j, -0.01273198+0.00414389j, -0.01204725+0.00597486j, ...,  0.00022375+0.00682789j,  0.00212203+0.00639483j,  0.00377646+0.00548359j],
       [ 0.00607938+0.00661749j,  0.00764766+0.00570824j,  0.00906296+0.00443277j, ...,  0.00216894-0.0012144j ,  0.0021677 -0.00177j   ,  0.00199682-0.00239705j],
       [-0.01943596+0.01879979j, -0.01353566+0.02340886j, -0.00659764+0.02622327j, ...,  0.02289399+0.01438937j,  0.02596224+0.00755981j,  0.02704008+0.00015067j]])

### 用 scipy 的 expm_multiply

In [5]:
res = sp.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100)
np.squeeze(res).T

array([[ 0.00335624+0.00606165j,  0.0048898 +0.00490894j,  0.00604849+0.00337989j, ...,  0.00508586-0.00470552j,  0.00360056-0.00591979j,  0.00183922-0.00668021j],
       [-0.01145467+0.0090065j , -0.00893959+0.01087129j, -0.00609528+0.01202976j, ...,  0.0022496 +0.01370704j,  0.0058218 +0.01275967j,  0.00901401+0.01092018j],
       [ 0.00826567+0.0048053j ,  0.01002091+0.00476051j,  0.01195179+0.00416268j, ..., -0.00079423+0.0112147j ,  0.00174312+0.01032498j,  0.00388811+0.00880588j],
       ...,
       [-0.01305153+0.00220341j, -0.01273198+0.00414389j, -0.01204725+0.00597486j, ...,  0.00022375+0.00682789j,  0.00212203+0.00639483j,  0.00377646+0.00548359j],
       [ 0.00607938+0.00661749j,  0.00764766+0.00570824j,  0.00906296+0.00443277j, ...,  0.00216894-0.0012144j ,  0.0021677 -0.00177j   ,  0.00199682-0.00239705j],
       [-0.01943596+0.01879979j, -0.01353566+0.02340886j, -0.00659764+0.02622327j, ...,  0.02289399+0.01438937j,  0.02596224+0.00755981j,  0.02704008+0.00015067j]])

### 使用 quante 的 expm_multiply

In [7]:
res = qt.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100)
np.squeeze(res).T

array([[ 0.00335624+0.00606165j,  0.0048898 +0.00490894j,  0.00604849+0.00337989j, ...,  0.00508586-0.00470552j,  0.00360056-0.00591979j,  0.00183922-0.00668021j],
       [-0.01145467+0.0090065j , -0.00893959+0.01087129j, -0.00609528+0.01202976j, ...,  0.0022496 +0.01370704j,  0.0058218 +0.01275967j,  0.00901401+0.01092018j],
       [ 0.00826567+0.0048053j ,  0.01002091+0.00476051j,  0.01195179+0.00416268j, ..., -0.00079423+0.0112147j ,  0.00174312+0.01032498j,  0.00388811+0.00880588j],
       ...,
       [-0.01305153+0.00220341j, -0.01273198+0.00414389j, -0.01204725+0.00597486j, ...,  0.00022375+0.00682789j,  0.00212203+0.00639483j,  0.00377646+0.00548359j],
       [ 0.00607938+0.00661749j,  0.00764766+0.00570824j,  0.00906296+0.00443277j, ...,  0.00216894-0.0012144j ,  0.0021677 -0.00177j   ,  0.00199682-0.00239705j],
       [-0.01943596+0.01879979j, -0.01353566+0.02340886j, -0.00659764+0.02622327j, ...,  0.02289399+0.01438937j,  0.02596224+0.00755981j,  0.02704008+0.00015067j]])

针对实矩阵的优化

In [9]:
res = qt.linalg.expm_multiply(mat, state, -1j, start=0, stop=10, num=100)
np.squeeze(res).T

array([[ 0.00335624+0.00606165j,  0.0048898 +0.00490894j,  0.00604849+0.00337989j, ...,  0.00508586-0.00470552j,  0.00360056-0.00591979j,  0.00183922-0.00668021j],
       [-0.01145467+0.0090065j , -0.00893959+0.01087129j, -0.00609528+0.01202976j, ...,  0.0022496 +0.01370704j,  0.0058218 +0.01275967j,  0.00901401+0.01092018j],
       [ 0.00826567+0.0048053j ,  0.01002091+0.00476051j,  0.01195179+0.00416268j, ..., -0.00079423+0.0112147j ,  0.00174312+0.01032498j,  0.00388811+0.00880588j],
       ...,
       [-0.01305153+0.00220341j, -0.01273198+0.00414389j, -0.01204725+0.00597486j, ...,  0.00022375+0.00682789j,  0.00212203+0.00639483j,  0.00377646+0.00548359j],
       [ 0.00607938+0.00661749j,  0.00764766+0.00570824j,  0.00906296+0.00443277j, ...,  0.00216894-0.0012144j ,  0.0021677 -0.00177j   ,  0.00199682-0.00239705j],
       [-0.01943596+0.01879979j, -0.01353566+0.02340886j, -0.00659764+0.02622327j, ...,  0.02289399+0.01438937j,  0.02596224+0.00755981j,  0.02704008+0.00015067j]])

## 比较运行速度

生成一个较大的矩阵

In [21]:
# 拿到矩阵
L = 18
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)
print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

space dimension: 262144


比较四种方法运行所需要的时间

In [22]:
# 方法1:
import scipy.sparse as sp

t = time.time()
res1 = sp.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100)
print(f"time scipy.expm_multiple: {time.time()-t:.2f}s")

t = time.time()
res2 = qt.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100, herm=True)
print(f"time quante.expm_multiple with cpu parallel: {time.time()-t:.2f}s")

t = time.time()
res3 = qt.linalg.expm_multiply((-1j*mat), state, start=0, stop=10, num=100, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

t = time.time()
res4 = qt.linalg.expm_multiply(mat, state, -1j, start=0, stop=10, num=100, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

print("norm difference between first two:", np.allclose(res1, res2))
print("norm difference between last two:", np.allclose(res2, res3))
print("norm difference between last two:", np.allclose(res3, res4))

time scipy.expm_multiple: 26.57s
time quante.expm_multiple with cpu parallel: 17.28s


e:\hzhu\onedrive\python_library\quante\torch_utils\linalg\sparse.py:21: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return tc.sparse_csr_tensor(tsr.indptr, tsr.indices, tsr.data, tsr.shape, dtype=dtype, device=device)


time quante.expm_multiple with gpu cuda: 3.25s
time quante.expm_multiple with gpu cuda: 1.37s
norm difference between first two: True
norm difference between last two: True
norm difference between last two: True


建一个更大的矩阵

In [3]:
L = 22
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)
mat = ham.to_matrix(basis, sparse=True)
print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

space dimension: 4194304


演化 1s 需要的时间

In [4]:
# 方法1:
t = time.time()
res1 = sp.linalg.expm_multiply((-1j*mat), state)
print(f"time scipy.expm_multiple: {time.time()-t:.2f}s")

t = time.time()
res2 = qt.linalg.expm_multiply((-1j*mat), state, herm=True)
print(f"time quante.expm_multiple with cpu parallel: {time.time()-t:.2f}s")

t = time.time()
res3 = qt.linalg.expm_multiply((-1j*mat), state, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

t = time.time()
res4 = qt.linalg.expm_multiply(mat, state, -1j, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

print("norm difference between first two:", np.allclose(res1, res2))
print("norm difference between last two:", np.allclose(res1, res3))
print("norm difference between last two:", np.allclose(res1, res4))

time scipy.expm_multiple: 15.20s
time quante.expm_multiple with cpu parallel: 9.33s


e:\hzhu\onedrive\python_library\quante\torch_utils\linalg\sparse.py:21: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return tc.sparse_csr_tensor(tsr.indptr, tsr.indices, tsr.data, tsr.shape, dtype=dtype, device=device)


time quante.expm_multiple with gpu cuda: 3.89s
time quante.expm_multiple with gpu cuda: 0.46s
norm difference between first two: True
norm difference between last two: True
norm difference between last two: True


所能运行的最大尺寸

In [3]:
# 拿到矩阵
L = 26
ham = qt.generate.operas.heisenberg_operator(L)
ham = ham.expandxy()
basis = qt.generate.basis.spin_basis(L)
t = time.time()
mat = ham.to_matrix(basis, sparse=True)
print(f"time to generate matrix: {time.time()-t:.2f}s")

print("space dimension:", basis.Ns)

# 拿到态
state = qt.generate.state.random(basis.Ns, seed=42)

t = time.time()
res3 = qt.linalg.expm_multiply(mat, state, scale=-1j, usecuda=True, herm=True)
print(f"time quante.expm_multiple with gpu cuda: {time.time()-t:.2f}s")

time to generate matrix: 39.80s
space dimension: 67108864
time quante.expm_multiple with gpu cuda: 8.49s


In [8]:
print(f"哈密顿量内存占用：",end="")
qt.basicfun.test_memory(mat)
print(f"演化态内存占用：",end="");
qt.basicfun.test_memory(res3)

哈密顿量内存占用：10.38 GB
演化态内存占用：1.00 GB


## 逐步演化形式

In [3]:
from quante.torch_utils.linalg import evolve_engine, to_csr # 只在 torch 中实现了
import torch as tc

L = 10
ham = qt.generate.operas.heisenberg_operator(L)
basis = qt.generate.basis.spin_basis(L)

# 生成矩阵和向量，并转为 torch tensor
tcmat = to_csr(ham.to_matrix(basis, sparse=True), device='cuda')
tcstate = tc.tensor(qt.generate.state.random(basis.Ns, seed=42), device='cuda')

# 定义演化引擎
eg = evolve_engine(tcmat, scale=-1j, herm=True)

# 进行演化
for i in range(10):
    nextstate = eg(tcstate)
    assert tc.allclose(nextstate, tc.matrix_exp((-1j)*tcmat.to_dense()) @ tcstate)
    
    # ... 做一些其他事情
    
    tcstate = nextstate

e:\hzhu\onedrive\python_library\quante\torch_utils\linalg\sparse.py:21: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return tc.sparse_csr_tensor(tsr.indptr, tsr.indices, tsr.data, tsr.shape, dtype=dtype, device=device)
